# Dataset Overview

This notebook documents the dataset used to train and evaluate the human-vs-AI text detector in this repository. It covers:

1. **The raw text corpus** — `data/dataset_ready_final/*.jsonl`. Humans, AI-original texts, and LLM rewrites.
2. **The training filter** — every sample's author must contribute at least 2 human texts as TRACE context. This makes the training set USE-only: HC3 humans (no author), HC3-AI, ArguGPT, RAID (model-as-author with no humans), and USE authors with too few essays are all excluded.
3. **The cached extractor features** — `data/features/*.npz`. NELA + StyleDecipher + TRACE feature triples consumed by training.
4. **How to unpack `features.tar.gz`** if you've just downloaded the built cache off a remote GPU.

For the full dataset-construction story (sources, rewrite generators, quality filtering, author-disjoint splitting), see [`README.md`](README.md). For training/evaluation, see [`../training/README.md`](../training/README.md).

## 1. The raw text corpus

The corpus combines three types of records, split **author-disjoint** into train/val/test so the TRACE author-fingerprint model cannot leak author identity across splits.

| Type | Label | Count | Description |
|---|---|---|---|
| Human texts | `human` | 3,415 | USE student essays + HC3 human answers |
| AI-original texts | `ai`    | 3,620 | ArguGPT, RAID, HC3 ChatGPT answers |
| LLM rewrites of human texts | `ai` | 6,051 | 5 LLMs paraphrasing USE essays (style-shift stress test) |
| **Total (raw `.jsonl`)** | | **13,086** | |

**Per-split totals in the raw `.jsonl` (before the known-author filter):**

| Split | Total | Human | AI |
|---|---|---|---|
| `train` | 9,265 | 2,406 | 6,859 |
| `val`   | 1,846 |   492 | 1,354 |
| `test`  | 1,975 |   517 | 1,458 |

Every record (in every `.jsonl`) shares the same schema:

| Field | Type | Meaning |
|---|---|---|
| `id` | `str` | unique ID (`h_` human, `a_` AI-original, `r_` rewrite) |
| `text` | `str` | the text content |
| `is_ai` / `label` | `bool` / `str` | redundant binary label |
| `source` | `str` | `use`, `use_rewrite`, `hc3`, `argugpt`, `raid` |
| `author_id` | `str?` | USE author (rewrites inherit their source's author) |
| `model` | `str?` | generating model for AI / rewrite records |
| `source_text_id` | `str?` | for rewrites: the source human record's `id` |
| `split` | `str` | `train` / `val` / `test` |

## 2. The known-author + ≥2-sibling filter

Two stacked filters narrow the raw corpus down to records suitable for TRACE-style author profiling:

1. **`--require-known-author`** — drops records whose author is unknown. HC3 humans were scraped from open Q&A and have no per-author identity (`human_author_id` and `generator` are both null), so the filter discards them.
2. **`--min-human-siblings 2`** — for each remaining sample, count the same-author HUMAN texts available as TRACE context (excluding the source-text of an LLM rewrite, to block original↔rewrite leakage). Drop the sample if that count is < 2.

Effect of the second filter:

* **HC3-AI, ArguGPT, RAID** all have a *generator* (model) as their `author_id` with no human siblings → 0 humans by that author → dropped.
* **USE singleton authors** (USE students who contributed only 1–2 essays) → dropped.
* **USE authors with ≥3 essays** → kept. Every kept sample has ≥2 humans to compare against in TRACE.

Net effect: **the training set is USE-only** (USE original essays + their LLM rewrites). Non-USE sources contribute nothing to training under this rule.

**Records dropped by the filters:**

| Split | Raw | After `--require-known-author` | After `--min-human-siblings 2` (kept) |
|---|---|---|---|
| `train` | 9,265 | 7,767 | **4,051** |
| `val`   | 1,846 | 1,526 |   **667** |
| `test`  | 1,975 | 1,650 |   **829** |

**Class balance.** Each USE author contributes ~3 human essays + ~15 LLM rewrites (5 LLMs × ~3 essays), so the human:AI ratio is consistently **~1:4.76** in every split. Training still uses inverse-frequency class weighting; the *strict-FPR* deployment regime is unaffected (it operates on probability ranking, not class balance).

## 3. The cached extractor features

Running the three feature extractors (NELA, StyleDecipher, TRACE) over every record is slow — StyleDecipher in particular calls 5 LLMs per non-cluster record. To make training fast and repeatable, `training/build_dataset.py` runs the extractors **once** and caches the result. The training scripts then never touch the extractors.

The cache lives at `data/features/`:

| File | Contents |
|---|---|
| `train.npz` | feature arrays for the filtered train split |
| `val.npz`   | feature arrays for the filtered val split |
| `test.npz`  | feature arrays for the filtered test split |
| `meta.json` | build configuration + per-split summary stats |

Each `.npz` contains row-aligned arrays:

| Array | Shape | Description |
|---|---|---|
| `nela`     | (N, 87)  | NELA surface / linguistic / credibility features |
| `style`    | (N, 10)  | StyleDecipher: mean + std of 5 similarity metrics vs LLM rewrites |
| `trace`    | (N, 128) | TRACE author-style embedding |
| `label`    | (N,)     | 0 = human, 1 = ai |
| `style_ok` | (N,)     | True if real StyleDecipher features were computed (False → zero vector) |
| `ids`      | (N,)     | record IDs from the source `.jsonl` (row ↔ record) |
| `sources`  | (N,)     | source corpus name (matches the `.jsonl` `source` field) |

**TRACE mode (`trace_context = "author"`).** TRACE produces an author-aware embedding instead of a single-text one. For each sample, the embedding is computed from:

* the sample's own text (always), plus
* the sample author's *other* HUMAN texts, **excluding** the source-text of an LLM rewrite (to block original↔rewrite leakage).

After the `--min-human-siblings 2` filter, the kept set is USE-only and every sample's TRACE context is genuinely author-aware:

| Source | Typical # of human siblings | TRACE context |
|---|---|---|
| `use` (human essay) | 2–4 | author's other USE essays |
| `use_rewrite` (LLM rewrite) | 2–4 | author's other USE essays, minus the source |

**StyleDecipher coverage note.** `style_ok` distinguishes records that actually got LLM rewrites compared against them from ones that didn't. In `--styledecipher ollama` mode (what this cache was built with), every record's text was sent to 5 LLMs that produced fresh rewrites, so `style_ok` should be ~100 % across every source corpus.

## 4. Extract `features.tar.gz`

If you just downloaded the built cache from a remote GPU as `features.tar.gz`, the next cell unpacks it into `data/features/`. Safe to re-run — it's a no-op if `data/features/` already exists.

In [ ]:
import tarfile
from pathlib import Path

# Resolve paths relative to this notebook (works whether you launch from repo root or data/)
NB_DIR       = Path.cwd() if Path.cwd().name == "data" else Path.cwd() / "data"
REPO_ROOT    = NB_DIR.parent
FEATURES_DIR = NB_DIR / "features"
TARBALL      = REPO_ROOT / "features.tar.gz"

if FEATURES_DIR.exists() and any(FEATURES_DIR.glob("*.npz")):
    print(f"already unpacked: {FEATURES_DIR}")
elif TARBALL.exists():
    print(f"extracting {TARBALL.name} → {REPO_ROOT}")
    with tarfile.open(TARBALL) as t:
        t.extractall(REPO_ROOT)
    print(f"done  → {FEATURES_DIR}")
else:
    raise FileNotFoundError(
        f"Neither {FEATURES_DIR} nor {TARBALL} exists.\n"
        f"Either rebuild the cache (see ../training/README.md) or place features.tar.gz at the repo root."
    )

print("\nfiles in data/features/:")
for p in sorted(FEATURES_DIR.iterdir()):
    print(f"  {p.name:<14} {p.stat().st_size/1024:>8.1f} KB")

## 5. Load the cache and read the build configuration

`meta.json` records exactly how the cache was built — the most important fields after the 2026-05-26 rebuild are `trace_context`, `require_known_author`, and `rebuild_note` (only present when a smart TRACE-only re-extraction was used to avoid re-running Ollama).

In [ ]:
import json
import numpy as np

meta = json.loads((FEATURES_DIR / "meta.json").read_text())

print("Build configuration:")
print(f"  styledecipher_mode   = {meta.get('styledecipher_mode')}")
print(f"  trace_context        = {meta.get('trace_context')}")
print(f"  require_known_author = {meta.get('require_known_author', False)}")
print(f"  min_human_siblings   = {meta.get('min_human_siblings', 0)}")
print(f"  dims                 = {meta['dims']}")
print(f"  seed                 = {meta.get('seed')}")
if meta.get('rebuild_note'):
    print(f"  rebuild_note         = {meta['rebuild_note']}")

print("\nPer-split summary (from meta.json):")
for sp, stats in meta["splits"].items():
    extras = []
    if stats.get("dropped_no_author"):
        extras.append(f"-{stats['dropped_no_author']} no-author")
    if stats.get("dropped_low_siblings"):
        extras.append(f"-{stats['dropped_low_siblings']} <{meta.get('min_human_siblings', 0)} sibs")
    extra = ("  (" + ", ".join(extras) + ")") if extras else ""
    print(f"  {sp:<5} records={stats['records']:>5}  human={stats['human']:>5}  ai={stats['ai']:>5}  "
          f"style_coverage={stats['style_coverage']:.1%}{extra}")

splits = {sp: np.load(FEATURES_DIR / f"{sp}.npz", allow_pickle=True) for sp in ("train", "val", "test")}
print("\nArrays per split:", list(splits["train"].files))

## 6. Per-extractor sanity check

Each row carries three feature vectors. Confirm each extractor populated real (non-zero) values across every split, and look at their numeric ranges.

**Important — all three extractors must contribute.** A zero NELA row means NELA crashed on that text; a zero StyleDecipher row means no rewrites were available; a zero TRACE row should never happen (TRACE always has at least the sample's own text to embed). If any extractor's non-zero rate falls below ~99 %, investigate before training.

In [ ]:
for sp in ("train", "val", "test"):
    d = splits[sp]
    n = len(d["ids"])
    print(f"\n=== {sp} ({n} records) ===")
    for name in ("nela", "style", "trace"):
        arr = d[name]
        nz  = int((arr != 0).any(axis=1).sum())
        print(f"  {name:<5} shape={str(arr.shape):<12} non-zero rows: {nz:>5}/{n}  "
              f"mean={arr.mean():+.4f}  std={arr.std():.4f}  min={arr.min():+.4f}  max={arr.max():+.4f}")

## 7. StyleDecipher coverage broken down by source corpus

Health check for the `ollama`-mode build. Every source — including the non-cluster ones (`hc3`, `argugpt`, `raid`) that require fresh Ollama generations — should hit ~100 % `style_ok`. Anything materially lower means Ollama was down or rejecting requests for some records during the build.

In [ ]:
from collections import defaultdict

for sp in ("train", "val", "test"):
    d = splits[sp]
    buckets = defaultdict(list)
    for s, ok in zip(d["sources"], d["style_ok"]):
        buckets[str(s)].append(bool(ok))
    print(f"\n--- {sp} ---")
    for src in sorted(buckets):
        oks = buckets[src]
        print(f"  {src:<14} {sum(oks):>5}/{len(oks):<5} = {sum(oks)/len(oks):.0%}")

## 8. Label distribution

After the `min_human_siblings=2` filter the kept set is USE-only — every author has ≥3 essays, contributing 1 human + 5 LLM rewrites per essay. The human:AI ratio is therefore a clean ~1:4.76 in every split:

| Split | Human | AI | Ratio |
|---|---|---|---|
| `train` | 703 | 3,348 | 1:4.76 |
| `val`   | 116 |   551 | 1:4.75 |
| `test`  | 144 |   685 | 1:4.76 |

Training still uses inverse-frequency class weighting on this ratio.

In [ ]:
for sp in ("train", "val", "test"):
    labels = splits[sp]["label"]
    human  = int((labels == 0).sum())
    ai     = int((labels == 1).sum())
    total  = len(labels)
    print(f"  {sp:<5} human={human:>5} ({human/total:.1%})   ai={ai:>5} ({ai/total:.1%})   total={total}")

## 9. TRACE sibling stats — how strong is the author context?

Under `min_human_siblings=2`, every kept sample's author contributes ≥2 human texts (after rewrite-source exclusion). The distribution shows that filter at work: no zero-sibling rows survive, and most samples have 2–4 siblings (USE authors mostly have 3 essays).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(REPO_ROOT))
from data.preprocessing import Dataset

def human_siblings(sample, ds):
    """Same rule the rebuild used: same-author humans, minus the source-text of rewrites."""
    out = []
    for s in ds._by_author.get(sample.author_id, []):
        if s.record_id == sample.record_id:           continue
        if s.label != "human":                         continue
        if sample.is_rewrite and s.record_id == sample.source_text_id: continue
        out.append(s)
    return out

MIN_SIBS = meta.get("min_human_siblings", 0)

for sp in ("train", "val", "test"):
    ds = Dataset.load(sp)
    by_source = {}
    for s in ds:
        if not s.has_known_author:
            continue
        n = len(human_siblings(s, ds))
        if n < MIN_SIBS:                                                # mirror the build-time filter
            continue
        by_source.setdefault(s.source, []).append(n)
    print(f"\n--- {sp} (kept samples only, filter min_human_siblings={MIN_SIBS}) ---")
    for src in sorted(by_source):
        ns = by_source[src]
        bucket_min = min(ns); bucket_max = max(ns)
        avg = sum(ns) / len(ns)
        print(f"  {src:<14} n={len(ns):>5}  human_siblings min/avg/max = {bucket_min}/{avg:.2f}/{bucket_max}")

## 10. Where to go next

The cache is ready to feed into training and evaluation. Everything below runs locally on CPU/GPU and never touches the extractors:

```bash
# (Re)build the cache from scratch — slow, calls Ollama for StyleDecipher rewrites
python -m training.build_dataset --splits all --styledecipher ollama \
                                --trace-context author \
                                --require-known-author --min-human-siblings 2

# Cheap rebuild that PRESERVES the Ollama-derived StyleDecipher + NELA values
# and only recomputes TRACE in author mode (used after switching trace_context).
# Uses --min-human-siblings 2 by default.
python -m training.rebuild_trace_author

# Neural fusion model — one strategy or sweep all four
python -m training.train --fusion-method gating
python -m training.train --fusion-method all

# Classical classifier — one backend or sweep all
python -m training.train_classical --classifier xgboost
python -m training.train_classical --classifier all

# Evaluate any trained checkpoint on the test split (.pt or .joblib auto-detected)
python -m test.evaluate --model models/test_models/fusion_gating.pt
```

See [`../training/README.md`](../training/README.md) for the full training framework documentation — classifier choices, hyper-parameters, fusion strategies, and per-extractor importance reporting for classical models.